# Customer Analytics Using Machine Learning

Analysis pipeline for the Online Shoppers Purchasing Intention dataset.

In [ ]:
import os


In [ ]:
import numpy as np


In [ ]:
import pandas as pd


In [ ]:
import matplotlib.pyplot as plt


In [ ]:
from sklearn.model_selection import (


In [ ]:
    train_test_split,


In [ ]:
    StratifiedKFold,


In [ ]:
    cross_validate,


In [ ]:
    learning_curve


In [ ]:
)


In [ ]:
from sklearn.compose import ColumnTransformer


In [ ]:
from sklearn.pipeline import Pipeline


In [ ]:
from sklearn.impute import SimpleImputer


In [ ]:
from sklearn.preprocessing import (


In [ ]:
    StandardScaler,


In [ ]:
    OneHotEncoder


In [ ]:
)


In [ ]:
from sklearn.ensemble import RandomForestClassifier


In [ ]:
from sklearn.svm import SVC


In [ ]:
from sklearn.metrics import (


In [ ]:
    accuracy_score,


In [ ]:
    precision_score,


In [ ]:
    recall_score,


In [ ]:
    f1_score,


In [ ]:
    roc_auc_score,


In [ ]:
    average_precision_score,


In [ ]:
    classification_report,


In [ ]:
    confusion_matrix


In [ ]:
)


In [ ]:
from fairlearn.metrics import MetricFrame


## 1. LOAD DATASET

In [ ]:
DATA_PATH = "online_shoppers_intention.csv"

# Load dataset
df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

print("\nFirst five rows:")
display(df.head())

## 2. BASIC DATA INFORMATION

In [ ]:
print("\nDataset information:")
print(df.info())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDataset shape:")
print(df.shape)

## 3. CREATE OUTPUT DIRECTORY

In [ ]:
OUTPUT_DIR = "part1_3_results"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("\nOutput directory created:", OUTPUT_DIR)

## 4. PREPARE TARGET VARIABLE

In [ ]:
df["Revenue"] = df["Revenue"].map({
    True: 1,
    False: 0
})


df = df.dropna(subset=["Revenue"])

df["Revenue"] = df["Revenue"].astype(int)

print("\nTarget distribution:")
print(df["Revenue"].value_counts())

print("\nTarget distribution (%):")
print(df["Revenue"].value_counts(normalize=True) * 100)

## 5. DEFINE FEATURES AND TARGET

In [ ]:
X = df.drop(columns=["Revenue"])
y = df["Revenue"]

print("\nNumber of features:", X.shape[1])
print("Number of observations:", X.shape[0])

## 6. IDENTIFY NUMERICAL AND CATEGORICAL FEATURES

In [ ]:
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("\nNumerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

## 7. PREPROCESSING

In [ ]:
# Numerical preprocessing:
# - Median imputation
# - Standardisation

numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)


# Categorical preprocessing:
# - Most frequent imputation
# - One-hot encoding

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)


# Combine preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numeric_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)

## 8. 80:20 STRATIFIED TRAIN-TEST SPLIT

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True))

## 9. RANDOM FOREST MODEL

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

rf_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            rf_model
        )
    ]
)

## 10. SVM MODEL

In [ ]:
svm_model = SVC(
    kernel="rbf",
    probability=True,
    class_weight="balanced",
    random_state=42
)

svm_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            svm_model
        )
    ]
)

## 11. TRAIN RANDOM FOREST

In [ ]:
print("\nTraining Random Forest...")

rf_pipeline.fit(
    X_train,
    y_train
)

print("Random Forest training completed.")

## 12. TRAIN SVM

In [ ]:
print("\nTraining SVM...")

svm_pipeline.fit(
    X_train,
    y_train
)

print("SVM training completed.")

## 13. FUNCTION FOR MODEL EVALUATION

In [ ]:
def evaluate_model(
    model,
    X_test,
    y_test,
    model_name
):

    # Predictions
    y_pred = model.predict(X_test)

    # Probability predictions
    y_prob = model.predict_proba(X_test)[:, 1]

    # Metrics
    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )

    roc_auc = roc_auc_score(
        y_test,
        y_prob
    )

    pr_auc = average_precision_score(
        y_test,
        y_prob
    )

    results = {
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC-AUC": roc_auc,
        "PR-AUC": pr_auc
    }

    return results, y_pred, y_prob

## 14. EVALUATE RANDOM FOREST

In [ ]:
rf_results, rf_predictions, rf_probabilities = evaluate_model(
    rf_pipeline,
    X_test,
    y_test,
    "Random Forest"
)

## 15. EVALUATE SVM

In [ ]:
svm_results, svm_predictions, svm_probabilities = evaluate_model(
    svm_pipeline,
    X_test,
    y_test,
    "SVM"
)

## 16. DISPLAY MODEL RESULTS

In [ ]:
model_results = pd.DataFrame([
    rf_results,
    svm_results
])

print("\n================================================")
print("TASK 1 MODEL RESULTS")
print("================================================")

display(
    model_results.round(4)
)


# Save results
model_results.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "online_shoppers_model_results.csv"
    ),
    index=False
)

## 17. CLASSIFICATION REPORT - RANDOM FOREST

In [ ]:
print("\n================================================")
print("RANDOM FOREST CLASSIFICATION REPORT")
print("================================================")

print(
    classification_report(
        y_test,
        rf_predictions,
        zero_division=0
    )
)

## 18. CLASSIFICATION REPORT - SVM

In [ ]:
print("\n================================================")
print("SVM CLASSIFICATION REPORT")
print("================================================")

print(
    classification_report(
        y_test,
        svm_predictions,
        zero_division=0
    )
)

## 19. CONFUSION MATRICES

In [ ]:
rf_cm = confusion_matrix(
    y_test,
    rf_predictions
)

svm_cm = confusion_matrix(
    y_test,
    svm_predictions
)

print("\nRandom Forest Confusion Matrix:")
print(rf_cm)

print("\nSVM Confusion Matrix:")
print(svm_cm)

## 20. 5-FOLD STRATIFIED CROSS-VALIDATION

In [ ]:
print("\n================================================")
print("5-FOLD STRATIFIED CROSS-VALIDATION")
print("================================================")

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "f1": "f1",
    "roc_auc": "roc_auc",
    "precision": "precision",
    "recall": "recall"
}

## 21. RANDOM FOREST CROSS-VALIDATION

In [ ]:
print("\nRunning Random Forest cross-validation...")

rf_cv = cross_validate(
    rf_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

rf_cv_results = {
    "Model": "Random Forest",

    "Mean F1":
        rf_cv["test_f1"].mean(),

    "F1 Std":
        rf_cv["test_f1"].std(),

    "Mean ROC-AUC":
        rf_cv["test_roc_auc"].mean(),

    "ROC-AUC Std":
        rf_cv["test_roc_auc"].std(),

    "Mean Precision":
        rf_cv["test_precision"].mean(),

    "Precision Std":
        rf_cv["test_precision"].std(),

    "Mean Recall":
        rf_cv["test_recall"].mean(),

    "Recall Std":
        rf_cv["test_recall"].std()
}

## 22. SVM CROSS-VALIDATION

In [ ]:
print("Running SVM cross-validation...")

svm_cv = cross_validate(
    svm_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

svm_cv_results = {
    "Model": "SVM",

    "Mean F1":
        svm_cv["test_f1"].mean(),

    "F1 Std":
        svm_cv["test_f1"].std(),

    "Mean ROC-AUC":
        svm_cv["test_roc_auc"].mean(),

    "ROC-AUC Std":
        svm_cv["test_roc_auc"].std(),

    "Mean Precision":
        svm_cv["test_precision"].mean(),

    "Precision Std":
        svm_cv["test_precision"].std(),

    "Mean Recall":
        svm_cv["test_recall"].mean(),

    "Recall Std":
        svm_cv["test_recall"].std()
}

## 23. DISPLAY CROSS-VALIDATION RESULTS

In [ ]:
cv_results_df = pd.DataFrame([
    rf_cv_results,
    svm_cv_results
])

print("\n================================================")
print("CROSS-VALIDATION RESULTS")
print("================================================")

display(
    cv_results_df.round(4)
)


# Save CV results
cv_results_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "online_shoppers_cross_validation_results.csv"
    ),
    index=False
)

## 24. LEARNING CURVE FUNCTION

In [ ]:
def generate_learning_curve(
    model,
    X_train,
    y_train,
    model_name
):

    print(
        f"\nGenerating learning curve for {model_name}..."
    )

    train_sizes, train_scores, validation_scores = learning_curve(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="f1",
        train_sizes=np.linspace(
            0.10,
            1.00,
            5
        ),
        n_jobs=-1
    )

    train_mean = train_scores.mean(
        axis=1
    )

    train_std = train_scores.std(
        axis=1
    )

    validation_mean = validation_scores.mean(
        axis=1
    )

    validation_std = validation_scores.std(
        axis=1
    )

    # Print values
    learning_results = pd.DataFrame({
        "Training Samples": train_sizes,
        "Training F1": train_mean,
        "Training F1 Std": train_std,
        "Validation F1": validation_mean,
        "Validation F1 Std": validation_std
    })

    print(
        f"\n{model_name} Learning Curve Values:"
    )

    display(
        learning_results.round(4)
    )

    # Save values
    safe_name = model_name.lower().replace(
        " ",
        "_"
    )

    learning_results.to_csv(
        os.path.join(
            OUTPUT_DIR,
            f"{safe_name}_learning_curve_results.csv"
        ),
        index=False
    )

    # Plot
    plt.figure(
        figsize=(8, 5)
    )

    plt.plot(
        train_sizes,
        train_mean,
        marker="o",
        label="Training F1"
    )

    plt.plot(
        train_sizes,
        validation_mean,
        marker="o",
        label="Validation F1"
    )

    plt.fill_between(
        train_sizes,
        train_mean - train_std,
        train_mean + train_std,
        alpha=0.15
    )

    plt.fill_between(
        train_sizes,
        validation_mean - validation_std,
        validation_mean + validation_std,
        alpha=0.15
    )

    plt.xlabel(
        "Number of Training Samples"
    )

    plt.ylabel(
        "F1 Score"
    )

    plt.title(
        f"{model_name} Learning Curve"
    )

    plt.legend()

    plt.grid(
        True,
        alpha=0.3
    )

    plt.tight_layout()

    figure_path = os.path.join(
        OUTPUT_DIR,
        f"{safe_name}_learning_curve.png"
    )

    plt.savefig(
        figure_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    return learning_results

## 25. RANDOM FOREST LEARNING CURVE

In [ ]:
rf_learning_results = generate_learning_curve(
    rf_pipeline,
    X_train,
    y_train,
    "Random Forest"
)

## 26. SVM LEARNING CURVE

In [ ]:
svm_learning_results = generate_learning_curve(
    svm_pipeline,
    X_train,
    y_train,
    "SVM"
)

## 27. FAIRNESS / SUBGROUP ANALYSIS

In [ ]:
print("\n================================================")
print("FAIRNESS / SUBGROUP ANALYSIS")
print("================================================")

print(
    """
VisitorType is used as the subgroup variable.
It represents behavioural visitor categories rather
than a protected demographic attribute.
"""
)

## 28. CHECK VISITOR TYPE GROUPS

In [ ]:
print("\nVisitorType groups in test data:")

print(
    X_test["VisitorType"].value_counts()
)


print("\nVisitorType proportions:")

print(
    X_test["VisitorType"].value_counts(
        normalize=True
    ).round(4)
)

## 29. RANDOM FOREST FAIRNESS ANALYSIS

In [ ]:
print("\nRunning Fairlearn analysis for Random Forest...")

rf_metric_frame = MetricFrame(
    metrics={
        "Recall": recall_score,
        "Precision": precision_score,
        "F1": f1_score
    },
    y_true=y_test,
    y_pred=rf_predictions,
    sensitive_features=X_test["VisitorType"]
)

## 30. DISPLAY RANDOM FOREST FAIRNESS RESULTS

In [ ]:
print("\n================================================")
print("RANDOM FOREST - SUBGROUP RESULTS")
print("================================================")

display(
    rf_metric_frame.by_group.round(4)
)


print("\nOverall Random Forest metrics:")

display(
    pd.DataFrame(
        rf_metric_frame.overall,
        index=["Overall"]
    ).round(4)
)

## 31. SAVE RANDOM FOREST FAIRNESS RESULTS

In [ ]:
rf_fairness_results = (
    rf_metric_frame
    .by_group
    .reset_index()
)

rf_fairness_results.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "random_forest_fairness_results.csv"
    ),
    index=False
)

## 32. SVM FAIRNESS ANALYSIS

In [ ]:
print("\nRunning Fairlearn analysis for SVM...")

svm_metric_frame = MetricFrame(
    metrics={
        "Recall": recall_score,
        "Precision": precision_score,
        "F1": f1_score
    },
    y_true=y_test,
    y_pred=svm_predictions,
    sensitive_features=X_test["VisitorType"]
)

## 33. DISPLAY SVM FAIRNESS RESULTS

In [ ]:
print("\n================================================")
print("SVM - SUBGROUP RESULTS")
print("================================================")

display(
    svm_metric_frame.by_group.round(4)
)


print("\nOverall SVM metrics:")

display(
    pd.DataFrame(
        svm_metric_frame.overall,
        index=["Overall"]
    ).round(4)
)

## 34. SAVE SVM FAIRNESS RESULTS

In [ ]:
svm_fairness_results = (
    svm_metric_frame
    .by_group
    .reset_index()
)

svm_fairness_results.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "svm_fairness_results.csv"
    ),
    index=False
)

## 35. COMBINED FAIRNESS TABLE

In [ ]:
rf_combined = (
    rf_metric_frame
    .by_group
    .reset_index()
)

rf_combined["Model"] = "Random Forest"

svm_combined = (
    svm_metric_frame
    .by_group
    .reset_index()
)

svm_combined["Model"] = "SVM"

combined_fairness = pd.concat(
    [
        rf_combined,
        svm_combined
    ],
    ignore_index=True
)

# Rearrange columns
combined_fairness = combined_fairness[
    [
        "Model",
        "VisitorType",
        "Recall",
        "Precision",
        "F1"
    ]
]

print("\n================================================")
print("COMBINED FAIRNESS RESULTS")
print("================================================")

display(
    combined_fairness.round(4)
)


# Save combined results
combined_fairness.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "combined_fairness_results.csv"
    ),
    index=False
)

## 36. FAIRNESS RANGE / DISPARITY SUMMARY

In [ ]:
print("\n================================================")
print("FAIRNESS DISPARITY SUMMARY")
print("================================================")


def calculate_disparity(
    metric_frame,
    model_name
):

    subgroup_results = metric_frame.by_group

    summary = {
        "Model": model_name,

        "Recall Max":
            subgroup_results["Recall"].max(),

        "Recall Min":
            subgroup_results["Recall"].min(),

        "Recall Difference":
            (
                subgroup_results["Recall"].max()
                -
                subgroup_results["Recall"].min()
            ),

        "Precision Max":
            subgroup_results["Precision"].max(),

        "Precision Min":
            subgroup_results["Precision"].min(),

        "Precision Difference":
            (
                subgroup_results["Precision"].max()
                -
                subgroup_results["Precision"].min()
            ),

        "F1 Max":
            subgroup_results["F1"].max(),

        "F1 Min":
            subgroup_results["F1"].min(),

        "F1 Difference":
            (
                subgroup_results["F1"].max()
                -
                subgroup_results["F1"].min()
            )
    }

    return summary


rf_disparity = calculate_disparity(
    rf_metric_frame,
    "Random Forest"
)

svm_disparity = calculate_disparity(
    svm_metric_frame,
    "SVM"
)

disparity_df = pd.DataFrame([
    rf_disparity,
    svm_disparity
])

display(
    disparity_df.round(4)
)


# Save disparity results
disparity_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "fairness_disparity_summary.csv"
    ),
    index=False
)

## 37. FINAL SUMMARY OF ALL OUTPUT FILES

In [ ]:
print("\n================================================")
print("ANALYSIS COMPLETED")
print("================================================")

print("\nFiles saved in:")
print(OUTPUT_DIR)

print("\nGenerated files:")

for filename in sorted(
    os.listdir(OUTPUT_DIR)
):
    print(
        " -",
        filename
    )

## 38. FINAL IMPORTANT RESULTS

In [ ]:
print("\n================================================")
print("IMPORTANT RESULTS FOR REPORT")
print("================================================")

print("\nTask 1 Model Results:")
display(
    model_results.round(4)
)

print("\n5-Fold Cross-Validation:")
display(
    cv_results_df.round(4)
)

print("\nRandom Forest Fairness Results:")
display(
    rf_fairness_results.round(4)
)

print("\nSVM Fairness Results:")
display(
    svm_fairness_results.round(4)
)

print("\nFairness Disparity Summary:")
display(
    disparity_df.round(4)
)

## 1. SVM LEARNING CURVE

In [ ]:
print("\n================================================")
print("SVM LEARNING CURVE")
print("================================================")

from sklearn.model_selection import learning_curve
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

train_sizes_svm, train_scores_svm, validation_scores_svm = learning_curve(
    svm_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring="f1",
    train_sizes=np.linspace(0.10, 1.00, 5),
    n_jobs=-1
)

# Calculate means and standard deviations
train_mean_svm = train_scores_svm.mean(axis=1)
train_std_svm = train_scores_svm.std(axis=1)

validation_mean_svm = validation_scores_svm.mean(axis=1)
validation_std_svm = validation_scores_svm.std(axis=1)

# Create results table
svm_learning_results = pd.DataFrame({
    "Training Samples": train_sizes_svm,
    "Training F1": train_mean_svm,
    "Training F1 Std": train_std_svm,
    "Validation F1": validation_mean_svm,
    "Validation F1 Std": validation_std_svm
})

print("\nSVM Learning Curve Values:")
display(svm_learning_results.round(4))

# Save results
svm_learning_results.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "svm_learning_curve_results.csv"
    ),
    index=False
)

# Plot
plt.figure(figsize=(8, 5))

plt.plot(
    train_sizes_svm,
    train_mean_svm,
    marker="o",
    label="Training F1"
)

plt.plot(
    train_sizes_svm,
    validation_mean_svm,
    marker="o",
    label="Validation F1"
)

plt.fill_between(
    train_sizes_svm,
    train_mean_svm - train_std_svm,
    train_mean_svm + train_std_svm,
    alpha=0.15
)

plt.fill_between(
    train_sizes_svm,
    validation_mean_svm - validation_std_svm,
    validation_mean_svm + validation_std_svm,
    alpha=0.15
)

plt.xlabel("Number of Training Samples")
plt.ylabel("F1 Score")
plt.title("SVM Learning Curve")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "svm_learning_curve.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 2. RANDOM FOREST FAIRNESS / SUBGROUP ANALYSIS

In [ ]:
print("\n================================================")
print("RANDOM FOREST - SUBGROUP RESULTS")
print("================================================")

from fairlearn.metrics import MetricFrame
from sklearn.metrics import (
    recall_score,
    precision_score,
    f1_score
)

# Random Forest predictions were already generated earlier
# using:
# rf_predictions = rf_pipeline.predict(X_test)

rf_metric_frame = MetricFrame(
    metrics={
        "Recall": recall_score,
        "Precision": precision_score,
        "F1": f1_score
    },
    y_true=y_test,
    y_pred=rf_predictions,
    sensitive_features=X_test["VisitorType"]
)

# Display subgroup results
print("\nRandom Forest performance by VisitorType:")

display(
    rf_metric_frame.by_group.round(4)
)

# Display overall performance
print("\nOverall Random Forest performance:")

display(
    pd.DataFrame(
        rf_metric_frame.overall,
        index=["Overall"]
    ).round(4)
)

# Save results
rf_fairness_results = (
    rf_metric_frame
    .by_group
    .reset_index()
)

rf_fairness_results.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "random_forest_fairness_results.csv"
    ),
    index=False
)

print(
    "\nRandom Forest fairness results saved successfully."
)

## 3. SVM FAIRNESS / SUBGROUP ANALYSIS

In [ ]:
print("\n================================================")
print("SVM - SUBGROUP RESULTS")
print("================================================")



svm_metric_frame = MetricFrame(
    metrics={
        "Recall": recall_score,
        "Precision": precision_score,
        "F1": f1_score
    },
    y_true=y_test,
    y_pred=svm_predictions,
    sensitive_features=X_test["VisitorType"]
)

# Displaying subgroup results
print("\nSVM performance by VisitorType:")

display(
    svm_metric_frame.by_group.round(4)
)

# Displaying overall performance
print("\nOverall SVM performance:")

display(
    pd.DataFrame(
        svm_metric_frame.overall,
        index=["Overall"]
    ).round(4)
)

# Save results
svm_fairness_results = (
    svm_metric_frame
    .by_group
    .reset_index()
)

svm_fairness_results.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "svm_fairness_results.csv"
    ),
    index=False
)

print(
    "\nSVM fairness results saved successfully."
)

## 4. FAIRNESS DISPARITY SUMMARY

In [ ]:
print("\n================================================")
print("FAIRNESS DISPARITY SUMMARY")
print("================================================")


def calculate_disparity(
    metric_frame,
    model_name
):

    subgroup_results = metric_frame.by_group

    summary = {
        "Model": model_name,

        "Recall Max":
            subgroup_results["Recall"].max(),

        "Recall Min":
            subgroup_results["Recall"].min(),

        "Recall Difference":
            (
                subgroup_results["Recall"].max()
                -
                subgroup_results["Recall"].min()
            ),

        "Precision Max":
            subgroup_results["Precision"].max(),

        "Precision Min":
            subgroup_results["Precision"].min(),

        "Precision Difference":
            (
                subgroup_results["Precision"].max()
                -
                subgroup_results["Precision"].min()
            ),

        "F1 Max":
            subgroup_results["F1"].max(),

        "F1 Min":
            subgroup_results["F1"].min(),

        "F1 Difference":
            (
                subgroup_results["F1"].max()
                -
                subgroup_results["F1"].min()
            )
    }

    return summary


# Random Forest disparity
rf_disparity = calculate_disparity(
    rf_metric_frame,
    "Random Forest"
)

# SVM disparity
svm_disparity = calculate_disparity(
    svm_metric_frame,
    "SVM"
)

# Create summary table
disparity_df = pd.DataFrame([
    rf_disparity,
    svm_disparity
])

print("\nFairness Disparity Summary:")

display(
    disparity_df.round(4)
)

# Save results
disparity_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "fairness_disparity_summary.csv"
    ),
    index=False
)

print(
    "\nFairness disparity results saved successfully."
)